# Data Cleaning & Split Verification

Run the data cleaning pipeline, verify splits, and save clean_splits.json to Google Drive.

**Issue #6: Data Cleaning (D2)**

## Setup: Clone Repo and Install Dependencies

In [ ]:
import os
import sys

# Clone or update RETINNA repository
REPO_DIR = '/content/RETINNA'

if os.path.exists(REPO_DIR):
    print("Updating repository...")
    os.chdir(REPO_DIR)
    os.system('git pull origin main')
else:
    print("Cloning RETINNA repository...")
    os.system('git clone https://github.com/scerruti/RETINNA.git /content/RETINNA')
    os.chdir(REPO_DIR)

print(f"✓ Repository ready at {REPO_DIR}")

# Install dependencies
%pip install -q torch torchvision matplotlib numpy scipy scikit-learn torchgeo

# Setup Colab environment and dataset caching
src_path = '/content/RETINNA/src'
if src_path not in sys.path:
    sys.path.append(src_path)

from colab_utils import setup_colab_environment, setup_cabuaur_cached

env = setup_colab_environment()
cabuaur_path = setup_cabuaur_cached(env)

print("✓ Environment ready")

## Run Data Cleaning Pipeline

In [ ]:
from data_cleaning import clean_dataset
import os

# Create output directory
os.makedirs('data', exist_ok=True)

# Run cleaning pipeline
splits = clean_dataset(
    dataset_root=cabuaur_path,
    output_path='data/clean_splits.json'
)

print("\n✅ Data cleaning complete")

## Verify Split Integrity

In [ ]:
import json

# Load and verify splits
with open('data/clean_splits.json', 'r') as f:
    splits_data = json.load(f)

print("\n" + "="*70)
print("SPLIT VERIFICATION")
print("="*70)

train_idx = set(splits_data['train'])
val_idx = set(splits_data['val'])
test_idx = set(splits_data['test'])

print(f"\nTrain set: {len(train_idx)} samples")
print(f"Val set:   {len(val_idx)} samples")
print(f"Test set:  {len(test_idx)} samples")
print(f"Total:     {len(train_idx) + len(val_idx) + len(test_idx)} samples")

# Check for overlaps
overlap_tv = train_idx & val_idx
overlap_tt = train_idx & test_idx
overlap_vt = val_idx & test_idx

print(f"\nOverlap checks:")
print(f"  Train-Val overlap: {len(overlap_tv)} (should be 0)")
print(f"  Train-Test overlap: {len(overlap_tt)} (should be 0)")
print(f"  Val-Test overlap: {len(overlap_vt)} (should be 0)")

# Check for gaps
all_indices = train_idx | val_idx | test_idx
min_idx = min(all_indices)
max_idx = max(all_indices)
expected_indices = set(range(len(splits_data['train']) + len(splits_data['val']) + len(splits_data['test'])))

# Just verify all are in valid range
print(f"\nIndex range: {min_idx} to {max_idx}")

if len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0:
    print("\n✅ PASS: No overlaps between splits")
else:
    print("\n❌ FAIL: Overlaps detected!")

print("\n" + "="*70)
print("Split ratios:")
print(f"  Train: {len(train_idx) / len(all_indices) * 100:.1f}%")
print(f"  Val:   {len(val_idx) / len(all_indices) * 100:.1f}%")
print(f"  Test:  {len(test_idx) / len(all_indices) * 100:.1f}%")
print("="*70)

## Test CaBuArCleanDataset Class

In [ ]:
from dataset import CaBuArCleanDataset

print("\nLoading datasets for each split...\n")

# Load each split
train_dataset = CaBuArCleanDataset(split='train', root=cabuaur_path, splits_file='data/clean_splits.json')
val_dataset = CaBuArCleanDataset(split='val', root=cabuaur_path, splits_file='data/clean_splits.json')
test_dataset = CaBuArCleanDataset(split='test', root=cabuaur_path, splits_file='data/clean_splits.json')

print(f"\n" + "="*70)
print("Dataset sizes:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")
print("="*70)

## Test Sample Access

In [ ]:
# Test getting a sample from each split
print("\nTesting sample access...\n")

for split_name, dataset in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    sample = dataset[0]
    image = sample['image']
    mask = sample['mask']
    
    print(f"{split_name.upper():5} split, sample 0:")
    print(f"  Image shape: {image.shape} (expected: [2, 12, 512, 512])")
    print(f"  Mask shape:  {mask.shape} (expected: [1, 512, 512])")
    print(f"  Image dtype: {image.dtype}")
    print(f"  Mask dtype:  {mask.dtype}")
    
    # Check for NaN
    has_nan_img = (image != image).any().item()  # NaN != NaN
    has_nan_mask = (mask != mask).any().item()
    
    print(f"  Contains NaN: image={has_nan_img}, mask={has_nan_mask}")
    print()

## Save clean_splits.json to Google Drive

In [ ]:
import shutil
from pathlib import Path

if env['drive_ready']:
    drive_data_dir = Path(env['drive_path']) / 'RETINNA_DATA'
    drive_data_dir.mkdir(parents=True, exist_ok=True)
    
    dest_path = drive_data_dir / 'clean_splits.json'
    shutil.copy('data/clean_splits.json', str(dest_path))
    
    print(f"✓ Saved clean_splits.json to {dest_path}")
    print(f"\nFuture notebooks can load it with:")
    print(f"  from src.dataset import CaBuArCleanDataset")
    print(f"  dataset = CaBuArCleanDataset(split='train', root=cabuaur_path)")
else:
    print("⚠ Google Drive not available - splits.json only saved locally")

## Summary